# Lab 05 — Thresholding & K-Means Color Segmentation

**Course:** Image Analysis and Processing — École des Mines de Saint-Étienne  
**Topics covered:**
1. Automatic global thresholding with Otsu's method
2. Custom iterative thresholding algorithm (ISODATA / Ridler-Calvard) vs. Otsu
3. 2D synthetic clustering with K-Means (`k-means++`)
4. 3D RGB color-space segmentation using K-Means

In [ ]:
import time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from skimage.io import imread
from skimage.filters import threshold_otsu
from sklearn.cluster import KMeans

IMAGES_DIR = Path("../Images")

## 1. Otsu Global Thresholding

We segment a grayscale microscopy image (`cells.bmp`) using Otsu's variance-maximization threshold and visualize the threshold on the intensity histogram.

In [ ]:
cells = imread(IMAGES_DIR / "cells.bmp", as_gray=True)
threshold = threshold_otsu(cells)
segmented_cells = cells > threshold

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cells, cmap="gray")
axes[0].set_title("Grayscale Image (cells.bmp)")
axes[0].axis("off")

axes[1].hist(cells.ravel(), bins=256, color="black")
axes[1].axvline(threshold, color="crimson", linestyle="--", label=f"Otsu T = {threshold:.3f}")
axes[1].set_title("Intensity Histogram")
axes[1].set_xlabel("Intensity")
axes[1].set_ylabel("Frequency")
axes[1].legend()

axes[2].imshow(segmented_cells, cmap="gray")
axes[2].set_title("Otsu Segmentation")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 2. Iterative Automatic Thresholding vs. Otsu

We implement the iterative mean-midpoint thresholding algorithm (Ridler-Calvard / ISODATA) from scratch and compare its threshold against `threshold_otsu`.

In [ ]:
def iterative_threshold(image: np.ndarray, epsilon: float = 0.5) -> float:
    """Iterative automatic thresholding (Ridler-Calvard algorithm)."""
    img = np.asarray(image, dtype=float)
    t = 0.5 * (np.amin(img) + np.amax(img))

    while True:
        foreground = img >= t
        t_next = 0.5 * (np.mean(img[foreground]) + np.mean(img[~foreground]))
        if abs(t - t_next) < epsilon:
            return float(t_next)
        t = t_next


manual_t = iterative_threshold(cells)
otsu_t = threshold_otsu(cells)

print(f"Iterative threshold: {manual_t:.3f}")
print(f"Otsu threshold:      {otsu_t:.3f}")
print(f"Absolute difference: {abs(manual_t - otsu_t):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(cells >= manual_t, cmap="gray")
axes[0].set_title(f"Iterative Thresholding (T = {manual_t:.3f})")
axes[0].axis("off")

axes[1].imshow(cells >= otsu_t, cmap="gray")
axes[1].set_title(f"Otsu Thresholding (T = {otsu_t:.3f})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 3. 2D K-Means Clustering on Synthetic Gaussian Blobs

Before clustering pixels in RGB space, we validate K-Means (`k-means++`) on three synthetic 2D Gaussian clusters.

In [ ]:
def generate_cluster(n: int, cx: float, cy: float, rng: np.random.Generator) -> np.ndarray:
    return rng.standard_normal((n, 2)) + np.array([cx, cy])


rng = np.random.default_rng(0)
pts = np.vstack([
    generate_cluster(100, 0, 0, rng),
    generate_cluster(100, 3, 4, rng),
    generate_cluster(100, -5, -3, rng),
])

n_clusters = 3
kmeans = KMeans(init="k-means++", n_clusters=n_clusters, n_init=10, random_state=0)
t0 = time.perf_counter()
kmeans.fit(pts)
elapsed = time.perf_counter() - t0

print(f"Computation time: {elapsed:.4f} s")
print("Cluster centers:\n", kmeans.cluster_centers_)

fig, ax = plt.subplots(figsize=(6, 5))
colors = ["#4EACC5", "#FF9C34", "#4E9A06"]
for k, col in enumerate(colors):
    members = kmeans.labels_ == k
    center = kmeans.cluster_centers_[k]
    ax.scatter(pts[members, 0], pts[members, 1], color=col, s=20, alpha=0.8, label=f"Cluster {k}")
    ax.scatter(center[0], center[1], color=col, edgecolor="black", marker="X", s=160)

ax.set_title("2D K-Means Clustering (k = 3)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. 3D RGB Color-Space Segmentation with K-Means

Each pixel is represented as a 3D point $(R, G, B)$, transforming the image into an $N \times 3$ feature matrix where $N$ is the total number of pixels. K-Means partitions the color space into $k = 3$ dominant color classes.

In [ ]:
image = imread(IMAGES_DIR / "Tv16.png")
display_image = image.copy()
image_f32 = image.astype(np.float32)

h, w, c = image_f32.shape
pixels = image_f32.reshape(h * w, c)

# Subsample for responsive 3D scatter visualization
sample_idx = rng.choice(pixels.shape[0], size=min(15000, pixels.shape[0]), replace=False)
sample = pixels[sample_idx]

fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(sample[:, 0], sample[:, 1], sample[:, 2], c=sample / 255.0, s=2, alpha=0.35)
ax.set_xlabel("Red")
ax.set_ylabel("Green")
ax.set_zlabel("Blue")
ax.set_title("3D RGB Distribution of Tv16.png")
plt.tight_layout()
plt.show()

k = 3
kmeans_color = KMeans(init="k-means++", n_clusters=k, n_init=10, random_state=0)
t0 = time.perf_counter()
kmeans_color.fit(pixels)
elapsed = time.perf_counter() - t0

labels_2d = kmeans_color.labels_.reshape(h, w)
centers = kmeans_color.cluster_centers_
segmented_rgb = np.clip(np.rint(centers[labels_2d]), 0, 255).astype(np.uint8)

print(f"Computation time: {elapsed:.4f} s")
print("Cluster centers (RGB):\n", np.round(centers, 1))

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(display_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

axes[1].imshow(segmented_rgb)
axes[1].set_title(f"K-Means Color Segmentation (k = {k})")
axes[1].axis("off")

plt.tight_layout()
plt.show()